# Segmentation-Guided ResNet-50 dengan 5-Fold CV di Google Colab

Notebook ini melatih **Segmentation-Guided ResNet-50** menggunakan citra parenkim dari `000_dataset_v2` dan probability map hasil U-Net dari eksperimen `a0d90f9e-3dd4-4de0-98af-12858696f613`. Tiga channel pertama berisi CT parenkim dan channel keempat berisi probability map sebagai guidance.

Dataset dan probability map disalin dari arsip di Google Drive ke penyimpanan lokal Colab. Hasil training dan salinan konfigurasi JSON ditulis langsung ke Google Drive.

Sebelum mulai, push kode terbaru ke branch yang dikonfigurasi, lalu pilih **Runtime > Change runtime type > GPU**. Jalankan seluruh cell secara berurutan. Progress bar `tqdm` akan terlihat saat copy, extraction, pemeriksaan data, training, validation, dan pengujian opsional.

> Satu `EXPERIMENT_ID` hanya dapat memiliki satu direktori `classification/guided_resnet50`. Training tidak akan menimpa hasil yang sudah ada.

## Persiapan arsip sebelum membuka Google Colab

Jalankan perintah berikut dari direktori utama repository. Arsip mempertahankan struktur `000_dataset_v2` dan `experiment_results` yang dibutuhkan oleh metadata serta konfigurasi training.

```bash
tar -czf a0d90f9e-3dd4-4de0-98af-12858696f613_guided_resnet50_parenchyma.tar.gz \
  000_dataset_v2/_segmentation_dataset/004_classification_cv_5fold_seed42.csv \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask \
  experiment_results/a0d90f9e-3dd4-4de0-98af-12858696f613/segmentation/unet/inference/probability_npy
```

Setelah selesai, upload arsip ke:

`MyDrive/mask-guided-lung-nodule-xai/a0d90f9e-3dd4-4de0-98af-12858696f613_guided_resnet50_parenchyma.tar.gz`

Probability map harus berasal dari eksperimen yang sama dengan `EXPERIMENT_ID`. Mask ground truth disertakan untuk visualisasi XAI opsional dan bukan input guidance.

## 1. Periksa GPU

In [ ]:
import shutil

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak tersedia. Aktifkan GPU pada pengaturan runtime Colab."
    )

disk = shutil.disk_usage("/content")
gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {gpu_memory:.1f} GiB")
print(f"Free disk: {disk.free / (1024**3):.1f} GiB")

## 2. Hubungkan Google Drive

Arsip dibaca dari Google Drive. Checkpoint, metrik, plot, dan konfigurasi eksperimen juga disimpan di sana.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Atur konfigurasi eksperimen

Ini adalah cell konfigurasi utama. Ubah lokasi repository, arsip, model, training, optimizer, atau DataLoader di sini sebelum menjalankan cell berikutnya. Kombinasi `CT_INPUT_TYPE='parenchyma'` dan `CT_PATH_COLUMN='ct_parenchyma_path'` harus tetap sesuai.

In [ ]:
from pathlib import Path

# Repository
REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Eksperimen
EXPERIMENT_ID = "a0d90f9e-3dd4-4de0-98af-12858696f613"
EXPERIMENT_COMPONENT = "classification/guided_resnet50"

# Dataset dan probability map
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
ARCHIVE_NAME = f"{EXPERIMENT_ID}_guided_resnet50_parenchyma.tar.gz"
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / ARCHIVE_NAME
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = Path("/content/guided_classification_training_data")
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
PROBABILITY_ROOT = (
    EXTRACTION_ROOT
    / "experiment_results"
    / EXPERIMENT_ID
    / "segmentation/unet/inference/probability_npy"
)
COPY_ARCHIVE_TO_LOCAL = True
FORCE_EXTRACT = False

# Output
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "experiment_results"
DRIVE_OUTPUT_DIR = (
    DRIVE_OUTPUT_ROOT / EXPERIMENT_ID / EXPERIMENT_COMPONENT
)

# Input data
METADATA_FILENAME = "004_classification_cv_5fold_seed42.csv"
CT_INPUT_TYPE = "parenchyma"
CT_PATH_COLUMN = "ct_parenchyma_path"
INPUT_HEIGHT = 224
INPUT_WIDTH = 224
NUM_FOLDS = 5
CLASS_TO_IDX = {"benign": 0, "malignant": 1}
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Model guided ResNet-50
PRETRAINED_WEIGHTS = "DEFAULT"
CLASSIFIER_DROPOUT = 0.3
ATTENTION_FUSION_STAGE = "layer3"
ATTENTION_FEATURE_CHANNELS = 1024
ATTENTION_HIDDEN_CHANNELS = 64
ATTENTION_ALPHA_INITIAL_VALUE = 0.0

# Training
NUM_EPOCHS = 100
BATCH_SIZE = 32  # Kurangi jika GPU kehabisan memori.
LEARNING_RATE = 1e-3
SEED = 42
TRANSFORM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.5
DEVICE = "auto"
AMP_ENABLED = False  # Training menggunakan full precision (FP32).

# Optimizer SGD
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NESTEROV = False

# Early stopping
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 0.0

# DataLoader
NUM_WORKERS = 2
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
PIN_MEMORY = True

# Pengujian opsional setelah training
RUN_TEST_AFTER_TRAINING = False
MAX_TEST_SAMPLES = None  # Gunakan 8 untuk smoke test.
TEST_BATCH_SIZE = 1
TEST_NUM_WORKERS = 0
TEST_DPI = 120

print(f"Dataset: {DATASET_ROOT}")
print(f"Probability map: {PROBABILITY_ROOT}")
print(f"Output: {DRIVE_OUTPUT_DIR}")

## 4. Clone atau perbarui repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    print("Memperbarui repository yang sudah ada...")
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    print("Mengunduh repository...")
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

training_script = (
    PROJECT_ROOT
    / "003_classification/segmentation_guided_cv_resnet50/train.py"
)
if not training_script.is_file():
    raise FileNotFoundError(f"Script training tidak ditemukan: {training_script}")

print(f"Repository siap: {PROJECT_ROOT}")

## 5. Instal package yang dibutuhkan

PyTorch dan Torchvision bawaan Colab dipertahankan agar sesuai dengan CUDA pada runtime aktif.

In [ ]:
import sys

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Semua dependency siap.")

## 6. Salin dan ekstrak arsip

Arsip disalin dahulu ke disk lokal Colab agar pembacaan data lebih stabil. Penyalinan dan ekstraksi menampilkan progress bar `tqdm`. Set `FORCE_EXTRACT=True` hanya ketika hasil ekstraksi perlu dibuat ulang.

In [ ]:
import tarfile

from tqdm.auto import tqdm


def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    """Salin satu file sambil menampilkan progress dalam byte."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".part")
    total_bytes = source.stat().st_size

    with source.open("rb") as input_file:
        with temporary_path.open("wb") as output_file:
            with tqdm(
                total=total_bytes,
                desc="Menyalin arsip",
                unit="B",
                unit_scale=True,
            ) as progress_bar:
                while True:
                    chunk = input_file.read(chunk_size)
                    if not chunk:
                        break
                    output_file.write(chunk)
                    progress_bar.update(len(chunk))

    temporary_path.replace(destination)


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Arsip tidak ditemukan: {DRIVE_ARCHIVE_PATH}")

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    drive_size = DRIVE_ARCHIVE_PATH.stat().st_size
    local_copy_is_current = (
        archive_path.is_file() and archive_path.stat().st_size == drive_size
    )

    if local_copy_is_current:
        print(f"Menggunakan arsip lokal: {archive_path}")
    else:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
else:
    archive_path = DRIVE_ARCHIVE_PATH

extraction_marker = EXTRACTION_ROOT / ".extraction_complete"

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if extraction_marker.is_file():
    print(f"Menggunakan data lokal: {EXTRACTION_ROOT}")
else:
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        for member in tqdm(members, desc="Mengekstrak data", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")

    extraction_marker.touch()

print(f"Data siap: {EXTRACTION_ROOT}")

## 7. Validasi CT, mask, dan probability map

Cell ini memeriksa struktur metadata, lima fold, seluruh CT parenkim, ground-truth mask, dan probability map. Setiap filename metadata harus mempunyai probability map dengan nama file yang sama.

In [ ]:
import csv

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_directories = [
    EXTRACTION_ROOT
    / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT
    / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT
    / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask",
    EXTRACTION_ROOT
    / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask",
    PROBABILITY_ROOT,
]
required_columns = {
    "dataset",
    "patient_id",
    "filename",
    CT_PATH_COLUMN,
    "mask_path",
    "label",
    "cv_group_id",
    "cv_nodule_id",
    "cv_role",
    "cv_fold",
}

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata tidak ditemukan: {metadata_path}")

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(f"Direktori tidak ditemukan: {directory}")

with metadata_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    column_names = set(reader.fieldnames or [])
    rows = list(reader)

missing_columns = required_columns - column_names
if missing_columns:
    raise ValueError(f"Kolom metadata tidak lengkap: {sorted(missing_columns)}")
if not rows:
    raise ValueError("Metadata tidak boleh kosong.")

missing_files = []
for row in tqdm(rows, desc="Memeriksa pasangan data", unit="sampel"):
    ct_path = DATASET_ROOT / row[CT_PATH_COLUMN]
    mask_path = DATASET_ROOT / row["mask_path"]
    probability_path = PROBABILITY_ROOT / Path(row["filename"]).name

    if not ct_path.is_file():
        missing_files.append(ct_path)
    if not mask_path.is_file():
        missing_files.append(mask_path)
    if not probability_path.is_file():
        missing_files.append(probability_path)

if missing_files:
    examples = "\n".join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(
        f"Ada {len(missing_files)} file yang tidak ditemukan.\n{examples}"
    )

development_folds = {
    int(row["cv_fold"])
    for row in rows
    if row["cv_role"].strip().lower() == "development"
}
if development_folds != set(range(NUM_FOLDS)):
    raise ValueError(f"Fold tidak sesuai: {sorted(development_folds)}")

label_counts = {}
for row in rows:
    label = row["label"].strip().lower()
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"Jumlah sampel: {len(rows):,}")
print(f"Distribusi kelas: {label_counts}")
print(f"Fold development: {sorted(development_folds)}")
print("Semua CT, mask, dan probability map tersedia.")

## 8. Buat dan simpan konfigurasi JSON

JSON dibuat langsung dari cell konfigurasi utama. Satu file disimpan di repository sementara untuk `train.py`, dan satu salinan persisten disimpan di Google Drive. Ketika training dimulai, konfigurasi efektif juga disalin ke direktori hasil.

In [ ]:
import json

if CT_INPUT_TYPE != "parenchyma":
    raise ValueError("Eksperimen ini harus memakai CT_INPUT_TYPE='parenchyma'.")
if CT_PATH_COLUMN != "ct_parenchyma_path":
    raise ValueError("Parenchyma harus memakai CT_PATH_COLUMN yang sesuai.")
if NUM_FOLDS != 5:
    raise ValueError("Metadata ini harus menggunakan NUM_FOLDS = 5.")
if BATCH_SIZE < 1 or NUM_EPOCHS < 1 or LEARNING_RATE <= 0:
    raise ValueError("Batch size, epoch, dan learning rate harus positif.")
if NUM_WORKERS < 0:
    raise ValueError("NUM_WORKERS tidak boleh negatif.")
if not 0.0 <= CLASSIFICATION_THRESHOLD <= 1.0:
    raise ValueError("CLASSIFICATION_THRESHOLD harus antara 0 dan 1.")
if DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("DEVICE harus 'auto', 'cpu', atau 'cuda'.")
if AMP_ENABLED:
    raise ValueError("AMP_ENABLED harus False untuk eksperimen ini.")

config = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "component": EXPERIMENT_COMPONENT,
    },
    "output": {
        "root_directory": str(DRIVE_OUTPUT_ROOT),
        "config_snapshot_filename": (
            "segmentation_guided_cv_resnet50.json"
        ),
    },
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "metadata_path": str(metadata_path),
        "ct_input_type": CT_INPUT_TYPE,
        "ct_path_column": CT_PATH_COLUMN,
        "probability_root": str(PROBABILITY_ROOT),
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "class_to_idx": CLASS_TO_IDX,
        "normalization_mean": NORMALIZATION_MEAN,
        "normalization_std": NORMALIZATION_STD,
    },
    "cross_validation": {
        "num_folds": NUM_FOLDS,
        "development_role": "development",
        "holdout_role": "holdout_test",
        "holdout_fold": -1,
        "group_column": "cv_group_id",
        "nodule_column": "cv_nodule_id",
        "fold_column": "cv_fold",
        "role_column": "cv_role",
    },
    "model": {
        "architecture": "SegmentationGuidedResNet50",
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "training_strategy": "full_fine_tuning",
        "trainable_component": "entire_model",
        "classifier_dropout": CLASSIFIER_DROPOUT,
        "attention_fusion_stage": ATTENTION_FUSION_STAGE,
        "attention_feature_channels": ATTENTION_FEATURE_CHANNELS,
        "attention_hidden_channels": ATTENTION_HIDDEN_CHANNELS,
        "attention_alpha_initial_value": (
            ATTENTION_ALPHA_INITIAL_VALUE
        ),
    },
    "training": {
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "transform_seed": TRANSFORM_SEED,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "device": DEVICE,
    },
    "optimizer": {
        "name": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": NESTEROV,
    },
    "dataloader": {
        "num_workers": NUM_WORKERS,
        "persistent_workers": PERSISTENT_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "pin_memory": PIN_MEMORY,
        "train_shuffle": True,
        "val_shuffle": False,
        "train_drop_last": False,
        "val_drop_last": False,
    },
    "early_stopping": {
        "enabled": True,
        "monitor": "val_loss",
        "mode": "min",
        "patience": EARLY_STOPPING_PATIENCE,
        "min_delta": EARLY_STOPPING_MIN_DELTA,
        "verbose": True,
        "restore_best_weights": True,
    },
    "amp": {"training_enabled": AMP_ENABLED},
    "checkpoint": {"save_latest": True},
}


def save_json(data, output_path):
    """Simpan dictionary sebagai JSON yang mudah dibaca."""

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)
        file.write("\n")


config_path = (
    PROJECT_ROOT
    / "003_classification/configs/segmentation_guided_cv_resnet50.json"
)
drive_config_path = (
    DRIVE_OUTPUT_ROOT
    / "saved_configs"
    / EXPERIMENT_ID
    / "segmentation_guided_cv_resnet50.json"
)
save_json(config, config_path)
save_json(config, drive_config_path)

print(json.dumps(config, indent=4))
print(f"Config repository: {config_path}")
print(f"Config Google Drive: {drive_config_path}")

## 9. Jalankan preflight check

Preflight mengambil satu pasangan CT dan probability map dengan transform validation yang sama seperti training. Bentuk input harus `[4, 224, 224]`: tiga channel CT dan satu channel probability.

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
dataset_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.dataset"
)
model_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.model"
)
transform_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.transforms"
)

validation_transform = transform_module.build_val_transform(
    height=INPUT_HEIGHT,
    width=INPUT_WIDTH,
    mean=tuple(NORMALIZATION_MEAN),
    std=tuple(NORMALIZATION_STD),
    seed=TRANSFORM_SEED,
)
validation_dataset = dataset_module.ProbabilityGuidedClassificationDataset(
    root_dir=DATASET_ROOT,
    metadata_path=metadata_path,
    split="val",
    cv_fold=0,
    probability_root=PROBABILITY_ROOT,
    ct_path_column=CT_PATH_COLUMN,
    class_to_idx=CLASS_TO_IDX,
    transform=validation_transform,
)
sample_input, sample_label = validation_dataset[0]

model = model_module.SegmentationGuidedResNet50(
    num_classes=len(CLASS_TO_IDX),
    dropout=CLASSIFIER_DROPOUT,
    weights=None,
    attention_hidden_channels=ATTENTION_HIDDEN_CHANNELS,
    attention_alpha_initial_value=ATTENTION_ALPHA_INITIAL_VALUE,
).to("cuda")
model.eval()

with torch.no_grad():
    sample_output = model(sample_input.unsqueeze(0).to("cuda"))

probability_channel = sample_input[3]
print(f"Validation fold 0: {len(validation_dataset):,} sampel")
print(f"Input shape: {tuple(sample_input.shape)}")
print(f"Output shape: {tuple(sample_output.shape)}")
print(f"Sample label: {int(sample_label)}")
print(f"Probability minimum: {probability_channel.min().item():.4f}")
print(f"Probability maximum: {probability_channel.max().item():.4f}")

del model
torch.cuda.empty_cache()

## 10. Mulai training 5-fold

Training dijalankan langsung dalam kernel notebook menggunakan `runpy`, sehingga progress bar `tqdm` untuk train dan validation terlihat di bawah cell. Setiap fold memakai model baru dan model terbaik dipilih berdasarkan validation loss.

In [ ]:
import runpy

if DRIVE_OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Output sudah ada: {DRIVE_OUTPUT_DIR}\n"
        "Ganti EXPERIMENT_ID untuk memulai training baru."
    )

print("Memulai Segmentation-Guided ResNet-50...", flush=True)
print(f"CT input: {CT_PATH_COLUMN}", flush=True)
print(f"Probability map: {PROBABILITY_ROOT}", flush=True)
print(f"Fold: {NUM_FOLDS}", flush=True)
print(f"Epoch per fold: {NUM_EPOCHS}", flush=True)
print(f"AMP enabled: {AMP_ENABLED}", flush=True)
print(f"Output: {DRIVE_OUTPUT_DIR}", flush=True)

original_arguments = sys.argv.copy()
try:
    sys.argv = [
        str(training_script),
        "--config",
        str(config_path),
    ]
    runpy.run_module(
        "003_classification.segmentation_guided_cv_resnet50.train",
        run_name="__main__",
    )
finally:
    sys.argv = original_arguments

## 11. Lihat hasil cross-validation

Cell ini menampilkan ringkasan setiap fold, plot gabungan, dan lokasi artefak utama.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary_path = DRIVE_OUTPUT_DIR / "cv_summary.csv"
plot_path = DRIVE_OUTPUT_DIR / "figures/cv_fold_metrics.png"
config_snapshot_path = (
    DRIVE_OUTPUT_DIR / "segmentation_guided_cv_resnet50.json"
)
oof_predictions_path = DRIVE_OUTPUT_DIR / "out_of_fold_predictions.csv"

if not summary_path.is_file():
    raise FileNotFoundError(f"Ringkasan training tidak ditemukan: {summary_path}")

display(pd.read_csv(summary_path))
if plot_path.is_file():
    display(Image(filename=str(plot_path), width=750))

print(f"Output directory: {DRIVE_OUTPUT_DIR}")
print(f"Config snapshot: {config_snapshot_path}")
print(f"OOF predictions: {oof_predictions_path}")
for fold in range(NUM_FOLDS):
    best_model_path = DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pth"
    print(f"Fold {fold} best model: {best_model_path}")

## 12. Pengujian holdout dan XAI (opsional)

Ubah `RUN_TEST_AFTER_TRAINING=True` pada cell konfigurasi untuk mengevaluasi ensemble lima model serta membuat Grad-CAM dan LRP. Gunakan `MAX_TEST_SAMPLES=8` untuk smoke test karena XAI seluruh holdout memerlukan waktu lama.

In [ ]:
if RUN_TEST_AFTER_TRAINING:
    test_module_name = (
        "003_classification.segmentation_guided_cv_resnet50.test"
    )
    test_arguments = [
        test_module_name,
        str(DRIVE_OUTPUT_DIR),
        "--batch-size",
        str(TEST_BATCH_SIZE),
        "--num-workers",
        str(TEST_NUM_WORKERS),
        "--device",
        DEVICE,
        "--dpi",
        str(TEST_DPI),
    ]

    if MAX_TEST_SAMPLES is not None:
        test_arguments.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

    original_arguments = sys.argv.copy()
    try:
        sys.argv = test_arguments
        runpy.run_module(test_module_name, run_name="__main__")
    finally:
        sys.argv = original_arguments

    print(f"Hasil test: {DRIVE_OUTPUT_DIR / 'test'}")
else:
    print("Pengujian dilewati karena RUN_TEST_AFTER_TRAINING=False.")

## Menjalankan eksperimen baru

Engine CV membuat satu direktori baru dan belum melanjutkan fold yang terputus secara otomatis. Untuk eksperimen baru, ubah `EXPERIMENT_ID`, buat arsip dengan ID yang sama, lalu jalankan kembali notebook. Jangan menghapus hasil lama sebelum memastikan semua artefak sudah tersimpan.